In [ ]:
!pip install -q sentence-transformers scikit-learn pygad


BERT, RoBERTa, MiniLM

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

from sentence_transformers import SentenceTransformer
import pygad

import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported.")

# === YOUR EXACT PATHS ===
COMMON_PATH = "/content/drive/MyDrive/Neha_PBL_2/Datasets/symbipredict_2022.csv"
RARE_PATH   = "/content/drive/MyDrive/Neha_PBL_2/Datasets/orphanet_rare_disease_symptoms.csv"

BASE_OUT  = "/content/drive/MyDrive/Neha_PBL_2"
DATA_OUT  = os.path.join(BASE_OUT, "Datasets")
EMB_OUT   = os.path.join(BASE_OUT, "Embeddings")
RES_OUT   = os.path.join(BASE_OUT, "Results")
MODEL_OUT = os.path.join(BASE_OUT, "Models")

for d in [DATA_OUT, EMB_OUT, RES_OUT, MODEL_OUT]:
    os.makedirs(d, exist_ok=True)

print("Output dirs ready:", DATA_OUT, EMB_OUT, RES_OUT, MODEL_OUT)


# === Helper functions ===

def normalize_symptom_name(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text.replace(" ", "_")


def freq_to_num(f):
    """Rough mapping from Orphanet frequency string to numeric probability."""
    if pd.isna(f):
        return 0.5
    s = str(f).lower()
    if "very frequent" in s:
        return 0.9
    if "frequent" in s and "very" not in s:
        return 0.6
    if "occasional" in s:
        return 0.2
    if "very rare" in s:
        return 0.03
    if "excluded" in s:
        return 0.0
    if "%" in s:
        try:
            return float(s.replace("%", ""))/100.0
        except Exception:
            return 0.5
    if "-" in s:
        try:
            a,b = s.split("-")
            return (float(a)+float(b))/200.0
        except Exception:
            return 0.5
    return 0.5


def fuzzy_memberships(p: float):
    """Triangular fuzzy memberships: low, medium, high over [0,1]."""
    p = float(max(0.0, min(1.0, p)))

    # low
    if p <= 0.5:
        mu_low = (0.5 - p) / 0.5
    else:
        mu_low = 0.0

    # high
    if p >= 0.5:
        mu_high = (p - 0.5) / 0.5
    else:
        mu_high = 0.0

    # medium
    if p <= 0.5:
        mu_med = p / 0.5
    else:
        mu_med = (1.0 - p) / 0.5

    return mu_low, mu_med, mu_high


print("Helper functions ready.")


Data loading
Cleaning
Normalization
Orphanet frequency mapping
Fuzzy logic functions
Project folder structure
ML libraries, embedding models, GA optimizer

In [ ]:
common = pd.read_csv(COMMON_PATH)
print("Raw common shape:", common.shape)

# Normalize column names
common.columns = [normalize_symptom_name(c) for c in common.columns]
assert "prognosis" in common.columns, "prognosis column missing in common data"

symptom_cols = [c for c in common.columns if c != "prognosis"]
common["prognosis"] = common["prognosis"].apply(normalize_symptom_name)

unique_diseases = sorted(common["prognosis"].unique())
label_map = {d:i for i,d in enumerate(unique_diseases)}
common["label"] = common["prognosis"].map(label_map)

X = common[symptom_cols].astype(float).values
y = common["label"].values

common_symptoms_norm = symptom_cols

print("Num symptoms:", len(symptom_cols))
print("Num diseases:", len(unique_diseases))
print("X shape:", X.shape)


prepare the entire common-disease dataset for embedding:
Loads dataset

Normalizes all names
Extracts symptoms
Normalizes prognosis names
Converts diseases → numeric labels
Builds X (input matrix)
Builds y (output labels)
Prints dataset dimensions




In [ ]:
rare = pd.read_csv(RARE_PATH)
print("Raw rare shape:", rare.shape)

# Expect columns: Disease_Name, Orpha_Code, HPO_ID, Symptom, Frequency
assert "Symptom" in rare.columns and "Frequency" in rare.columns, "Symptom/Frequency missing"

rare["symptom_norm"] = rare["Symptom"].apply(normalize_symptom_name)
rare["freq_numeric"] = rare["Frequency"].apply(freq_to_num)
rare = rare[rare["symptom_norm"] != ""].copy()

rare_symptoms_norm = sorted(rare["symptom_norm"].unique())

print("Clean rare shape:", rare.shape)
print("Unique rare diseases:", rare["Disease_Name"].nunique())
print("Unique normalized rare symptoms:", len(rare_symptoms_norm))


Load Orphanet dataset
Clean symptom names
Convert frequency labels → numeric
Remove invalid rows
Create list of unique rare symptoms
Print dataset statistics

This prepares the rare-disease dataset for:

Neural embedding
Similarity comparison
Fuzzy scoring
Hybrid ranking
Rare→Common disease mapping

In [ ]:
MANUAL_MEDICAL_TO_COMMON = {
    # pain / musculoskeletal
    "arthralgia": "joint_pain",
    "arthritis": "joint_pain",
    "myalgia": "muscle_pain",
    "backache": "back_pain",
    "lumbago": "back_pain",
    "dorsalgia": "back_pain",
    "arthropathy": "joint_pain",
    "coxalgia": "hip_joint_pain" if "hip_joint_pain" in common_symptoms_norm else "joint_pain",

    # headache / neuro
    "cephalalgia": "headache",
    "migraine": "headache",
    "tension_headache": "headache",
    "cluster_headache": "headache",

    # vomiting / nausea / GI
    "emesis": "vomiting",
    "nausea_and_vomiting": "vomiting",
    "projectile_vomiting": "vomiting",
    "bilious_vomiting": "vomiting",
    "hematemesis": "vomiting",
    "recurrent_vomiting": "vomiting",
    "nausea": "nausea" if "nausea" in common_symptoms_norm else "vomiting",

    # diarrhoea / constipation
    "diarrhea": "diarrhoea",
    "diarrhoea": "diarrhoea",
    "chronic_diarrhea": "diarrhoea",
    "steatorrhea": "diarrhoea",
    "constipation": "constipation",

    # fever / pyrexia
    "pyrexia": "high_fever" if "high_fever" in common_symptoms_norm else "mild_fever",
    "fever": "high_fever" if "high_fever" in common_symptoms_norm else "mild_fever",
    "recurrent_fever": "high_fever",
    "low_grade_fever": "mild_fever",

    # respiratory
    "cough": "cough",
    "productive_cough": "cough",
    "dry_cough": "cough",
    "dyspnea": "breathlessness" if "breathlessness" in common_symptoms_norm else "shortness_of_breath",
    "shortness_of_breath": "breathlessness" if "breathlessness" in common_symptoms_norm else "shortness_of_breath",
    "wheezing": "wheezing",
    "stridor": "breathlessness",

    # skin / rash / itching
    "pruritus": "itching",
    "urticaria": "skin_rash",
    "eczema": "skin_rash",
    "erythematous_rash": "skin_rash",
    "maculopapular_rash": "skin_rash",
    "petechiae": "skin_rash",

    # neuro / seizures / cognition
    "seizure": "convulsions" if "convulsions" in common_symptoms_norm else "headache",
    "epileptic_seizure": "convulsions" if "convulsions" in common_symptoms_norm else "headache",
    "tonic_clonic_seizure": "convulsions" if "convulsions" in common_symptoms_norm else "headache",
    "absence_seizure": "loss_of_consciousness" if "loss_of_consciousness" in common_symptoms_norm else "dizziness",
    "intellectual_disability": "lack_of_concentration" if "lack_of_concentration" in common_symptoms_norm else None,
    "sleep_abnormality": "altered_sensorium" if "altered_sensorium" in common_symptoms_norm else None,

    # fatigue / weakness
    "fatigability": "fatigue",
    "fatigue": "fatigue",
    "malaise": "fatigue",
    "asthenia": "fatigue",
    "muscle_weakness": "weakness_in_limbs" if "weakness_in_limbs" in common_symptoms_norm else "fatigue",
    "hypotonia": "weakness_in_limbs" if "weakness_in_limbs" in common_symptoms_norm else "fatigue",

    # GI pain / abdomen
    "abdominal_pain": "abdominal_pain",
    "stomach_ache": "stomach_pain",
    "gastralgia": "stomach_pain",
    "colicky_abdominal_pain": "abdominal_pain",

    # appetite / weight
    "loss_of_appetite": "loss_of_appetite" if "loss_of_appetite" in common_symptoms_norm else "weight_loss",
    "anorexia": "loss_of_appetite" if "loss_of_appetite" in common_symptoms_norm else "weight_loss",
    "weight_loss": "weight_loss",
    "failure_to_thrive": "weight_loss",

    # eye / vision
    "blurred_vision": "blurred_and_distorted_vision" if "blurred_and_distorted_vision" in common_symptoms_norm else None,
    "photophobia": "redness_of_eyes" if "redness_of_eyes" in common_symptoms_norm else None,
    "pain_behind_the_eyes": "pain_behind_the_eyes" if "pain_behind_the_eyes" in common_symptoms_norm else None,

    # liver / jaundice
    "jaundice": "yellowing_of_eyes" if "yellowing_of_eyes" in common_symptoms_norm else "yellow_urine",
    "hepatomegaly": "swelling_of_stomach" if "swelling_of_stomach" in common_symptoms_norm else None,
    "acute_liver_failure": "acute_liver_failure" if "acute_liver_failure" in common_symptoms_norm else None,

    # edema / swelling
    "peripheral_edema": "swelling_of_stomach" if "swelling_of_stomach" in common_symptoms_norm else None,
    "anasarca": "fluid_overload" if "fluid_overload" in common_symptoms_norm else None,
    "lymphadenopathy": "swelled_lymph_nodes" if "swelled_lymph_nodes" in common_symptoms_norm else None,

    # urinary
    "dysuria": "burning_micturition",
    "hematuria": "spotting__urination" if "spotting__urination" in common_symptoms_norm else "burning_micturition",
    "pollakiuria": "increased_urination" if "increased_urination" in common_symptoms_norm else "burning_micturition",

    # chest pain / cardiac
    "chest_pain": "chest_pain",
    "angina_pectoris": "chest_pain"
}
print("Manual medical→common mapping ready. Size:", len(MANUAL_MEDICAL_TO_COMMON))


The MANUAL_MEDICAL_TO_COMMON dictionary:

converts medical terminology → layman/common symptom names
ensures rare dataset aligns with common dataset
improves similarity, embeddings, and GA accuracy
handles synonyms, abbreviations, and variations
ensures robust rare-disease to common-disease matching

In [ ]:
def token_overlap_score(rare_sym, common_sym):
    rt = set(rare_sym.split("_"))
    ct = set(common_sym.split("_"))
    return len(rt & ct)

def heuristic_token_match(rare_sym, common_symptoms_norm, min_overlap=1):
    best_match = None
    best_score = 0

    for cs in common_symptoms_norm:
        score = token_overlap_score(rare_sym, cs)
        if score > best_score:
            best_score = score
            best_match = cs

    return best_match if best_score >= min_overlap else None

print("Heuristic token matching helpers ready.")


The token-based heuristic matcher:

breaks symptoms into tokens

finds overlap between rare/common symptoms

picks the best match

acts as an automatic symptom mapper

improves rare–common alignment when manual mapping is missing

makes your hybrid scoring + embeddings work better

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

rare_readable   = [s.replace("_"," ") for s in rare_symptoms_norm]
common_readable = [s.replace("_"," ") for s in common_symptoms_norm]

print("Encoding symptom texts for embeddings...")
emb_rare   = model.encode(rare_readable, normalize_embeddings=True, batch_size=64, show_progress_bar=True)
emb_common = model.encode(common_readable, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

def embedding_match_all(sim_threshold=0.5):
    sim = cosine_similarity(emb_rare, emb_common)
    mapping = {}
    for i, r in enumerate(rare_symptoms_norm):
        j = int(np.argmax(sim[i]))
        best_sim = float(sim[i][j])
        mapping[r] = (common_symptoms_norm[j], best_sim) if best_sim >= sim_threshold else (None, best_sim)
    return mapping

embedding_mapping = embedding_match_all(sim_threshold=0.5)
print("Embedding-based mapping computed.")


Handles synonyms

Handles long phrases

Handles unseen medical terms

Most flexible and intelligent matching

In [ ]:
def build_final_mapping(
    rare_symptoms_norm,
    common_symptoms_norm,
    manual_dict,
    embedding_mapping,
    token_min_overlap=1,
    embedding_min_sim=0.5
):
    common_set = set(common_symptoms_norm)
    final_map = {}

    for r in rare_symptoms_norm:

        # 1) manual
        if r in manual_dict and manual_dict[r] is not None:
            final_map[r] = manual_dict[r]
            continue

        # 2) exact match
        if r in common_set:
            final_map[r] = r
            continue

        # 3) token-overlap heuristic
        heuristic_val = heuristic_token_match(r, common_symptoms_norm, min_overlap=token_min_overlap)
        if heuristic_val:
            final_map[r] = heuristic_val
            continue

        # 4) embedding-based
        if r in embedding_mapping:
            cand, sim = embedding_mapping[r]
            if cand is not None and sim >= embedding_min_sim:
                final_map[r] = cand
                continue

        # 5) nothing works
        final_map[r] = None

    return final_map

final_mapping = build_final_mapping(
    rare_symptoms_norm,
    common_symptoms_norm,
    MANUAL_MEDICAL_TO_COMMON,
    embedding_mapping,
    token_min_overlap=1,
    embedding_min_sim=0.5
)

with open(os.path.join(DATA_OUT, "rare_to_common_mapping_FINAL.json"), "w") as f:
    json.dump(final_mapping, f, indent=2)

print("Saved FINAL rare→common mapping. Mapped symptoms:", sum(1 for v in final_mapping.values() if v is not None))


In [ ]:
# Fix: Remove manual/embedding mappings that point to symptoms not present in common dataset
valid_symptoms = set(symptom_cols)

cleaned_mapping = {}
for r, c in final_mapping.items():
    if c in valid_symptoms:
        cleaned_mapping[r] = c
    else:
        cleaned_mapping[r] = None     # mark it unmapped
        # debug print
        print(f"⚠️ Removing invalid mapping: {r} → {c}")

final_mapping = cleaned_mapping

print("Valid mapping count:", sum(1 for v in final_mapping.values() if v is not None))


In [ ]:
# COMMON: disease × symptom empirical frequencies (mean over patients)
common_mat = common.groupby("prognosis")[symptom_cols].mean()
common_mat.to_csv(os.path.join(DATA_OUT, "common_final_aligned_norm.csv"))
print("common_mat shape:", common_mat.shape)

# RARE: disease × common-symptom frequencies
rare_diseases = sorted(rare["Disease_Name"].unique())
rare_mat = pd.DataFrame(0.0, index=rare_diseases, columns=symptom_cols)

for _, row in rare.iterrows():
    dname = row["Disease_Name"]
    rsym  = row["symptom_norm"]
    mapped = final_mapping.get(rsym, None)
    if mapped:
        rare_mat.loc[dname, mapped] = max(rare_mat.loc[dname, mapped], row["freq_numeric"])

rare_mat.to_csv(os.path.join(DATA_OUT, "rare_final_aligned_norm.csv"))

print("rare_mat shape:", rare_mat.shape)


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train)),
    batch_size=64,
    shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val)),
    batch_size=64,
    shuffle=False
)

class DiseaseNet(nn.Module):
    def __init__(self, in_dim, h1=256, h2=128, emb_dim=64, out_dim=10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(h2, emb_dim),
            nn.ReLU()
        )
        self.classifier = nn.Linear(emb_dim, out_dim)

    def forward(self, x):
        z = self.encoder(x)
        logits = self.classifier(z)
        return logits, z

net = DiseaseNet(len(symptom_cols), out_dim=len(unique_diseases))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

for epoch in range(20):
    net.train()
    total_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        logits, _ = net(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # validation
    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for bx, by in val_loader:
            logits, _ = net(bx)
            preds = logits.argmax(1)
            correct += (preds == by).sum().item()
            total += len(by)
    print(f"Epoch {epoch+1:02d}  TrainLoss={total_loss:.3f}  ValAcc={correct/total:.4f}")

torch.save(net.state_dict(), os.path.join(MODEL_OUT, "common_disease_mlp.pth"))
print("Model saved to:", os.path.join(MODEL_OUT, "common_disease_mlp.pth"))


In [ ]:
# reload matrices just to be safe
common_mat = pd.read_csv(os.path.join(DATA_OUT, "common_final_aligned_norm.csv"), index_col=0)
rare_mat   = pd.read_csv(os.path.join(DATA_OUT, "rare_final_aligned_norm.csv"), index_col=0)

net.eval()

common_embeddings = {}
with torch.no_grad():
    for d, row in common_mat.iterrows():
        x = torch.tensor(row.values, dtype=torch.float32).unsqueeze(0)
        _, z = net(x)
        common_embeddings[d] = z.squeeze(0).numpy()

rare_embeddings = {}
with torch.no_grad():
    for d, row in rare_mat.iterrows():
        x = torch.tensor(row.values, dtype=torch.float32).unsqueeze(0)
        _, z = net(x)
        rare_embeddings[d] = z.squeeze(0).numpy()

np.save(os.path.join(EMB_OUT, "common_embeddings.npy"), common_embeddings)
np.save(os.path.join(EMB_OUT, "rare_embeddings.npy"), rare_embeddings)

print("Saved disease embeddings.")


In [ ]:
common_emb = np.load(os.path.join(EMB_OUT, "common_embeddings.npy"), allow_pickle=True).item()
rare_emb   = np.load(os.path.join(EMB_OUT, "rare_embeddings.npy"), allow_pickle=True).item()

common_names = list(common_emb.keys())
rare_names   = list(rare_emb.keys())

A = np.vstack([common_emb[d] for d in common_names])
B = np.vstack([rare_emb[d] for d in rare_names])

sim_matrix = cosine_similarity(B, A)

zero_shot = {}
for i, r in enumerate(rare_names):
    sims = sim_matrix[i]
    top_idx = sims.argsort()[::-1][:5]
    zero_shot[r] = [(common_names[j], float(sims[j])) for j in top_idx]

with open(os.path.join(RES_OUT, "zero_shot_similarity.json"), "w") as f:
    json.dump(zero_shot, f, indent=2)

print("Saved zero_shot_similarity.json")


In [ ]:
# ============================================
# STEP 13 — Fuzzy Logic + Genetic Algorithm
# ============================================

print("Running Step 13: Fuzzy Logic + Genetic Algorithm Optimization")

# --------------------------------------------
# 1. Compute fuzzy score for each disease
# --------------------------------------------

def disease_fuzzy_score(row, Wl, Wm, Wh):
    """
    Fuzzy scoring of a disease's symptom frequency profile.
    row : pandas Series of symptom frequencies for one disease.
    """
    total = 0.0
    for p in row.values:
        mu_low, mu_med, mu_high = fuzzy_memberships(p)
        total += mu_low*Wl + mu_med*Wm + mu_high*Wh
    return total

# Initial fuzzy weights
W_low_init, W_med_init, W_high_init = 0.2, 0.5, 0.8

# Compute fuzzy scores for all rare and common diseases
rare_fuzzy_scores = {
    d: disease_fuzzy_score(rare_mat.loc[d], W_low_init, W_med_init, W_high_init)
    for d in rare_mat.index
}

common_fuzzy_scores = {
    d: disease_fuzzy_score(common_mat.loc[d], W_low_init, W_med_init, W_high_init)
    for d in common_mat.index
}

print("Fuzzy scores computed.")

# --------------------------------------------
# 2. Prepare matrices for GA optimization
# --------------------------------------------

rare_idx_names   = list(rare_mat.index)
common_idx_names = list(common_mat.index)

A2 = np.vstack([common_embeddings[d] for d in common_idx_names])  # common embeddings
B2 = np.vstack([rare_embeddings[d] for d in rare_idx_names])      # rare embeddings

# Cosine similarity between rare & common embeddings
cos_rc = cosine_similarity(B2, A2)

# Rare fuzzy scores as vector
rare_fuzzy_vec = np.array([rare_fuzzy_scores[d] for d in rare_idx_names])

print("Matrices ready for GA.")

# --------------------------------------------
# 3. PyGAD Fitness Function (PyGAD >= 2.20.0)
# --------------------------------------------

def ga_fitness(ga_instance, solution, solution_idx):
    """
    solution = [alpha, beta, gamma, W_low, W_med, W_high]
    Fitness = sum over all rare diseases of:
              max( alpha*cosine + beta*fuzzy + gamma*embedding_norm )
    """
    alpha, beta, gamma, Wl, Wm, Wh = solution

    total_score = 0.0

    for i in range(len(rare_idx_names)):

        sims = cos_rc[i]                     # cosine similarities
        fuzzy_r = rare_fuzzy_vec[i]          # fuzzy score for this rare disease
        emb_norm = np.linalg.norm(B2[i])     # embedding norm for rare disease

        # Compute fuzzy part again with optimized weights
        mu_low, mu_med, mu_high = fuzzy_memberships(fuzzy_r)
        fuzzy_weighted = mu_low*Wl + mu_med*Wm + mu_high*Wh

        # Hybrid similarity
        hybrid = alpha*sims + beta*fuzzy_weighted + gamma*emb_norm

        # Add best matching score
        total_score += np.max(hybrid)

    return total_score

print("Fitness function ready.")

# --------------------------------------------
# 4. Run Genetic Algorithm
# --------------------------------------------

ga = pygad.GA(
    num_generations=15,
    num_parents_mating=6,
    sol_per_pop=12,
    num_genes=6,
    fitness_func=ga_fitness,
    init_range_low=0.0,
    init_range_high=1.0,
    mutation_percent_genes=20,
    parent_selection_type="tournament",
    keep_parents=2
)

ga.run()

print("GA finished.")

# --------------------------------------------
# 5. Extract Best Solution
# --------------------------------------------

solution, fitness, details = ga.best_solution()

alpha_opt, beta_opt, gamma_opt, W_low_opt, W_med_opt, W_high_opt = solution

print("========================================")
print("Best GA Parameters (Optimized):")
print("alpha     =", alpha_opt)
print("beta      =", beta_opt)
print("gamma     =", gamma_opt)
print("W_low     =", W_low_opt)
print("W_med     =", W_med_opt)
print("W_high    =", W_high_opt)
print("Fitness   =", fitness)
print("========================================")

# Save optimized parameters
opt_params = {
    "alpha": float(alpha_opt),
    "beta": float(beta_opt),
    "gamma": float(gamma_opt),
    "W_low": float(W_low_opt),
    "W_med": float(W_med_opt),
    "W_high": float(W_high_opt),
    "fitness": float(fitness)
}

with open(os.path.join(RES_OUT, "optimized_ga_parameters.json"), "w") as f:
    json.dump(opt_params, f, indent=2)

print("Saved optimized GA parameters to optimized_ga_parameters.json")


In [ ]:
# ============================================
# STEP 14 — Hybrid ML + Fuzzy + GA Inference
# ============================================

print("Running Step 14: Hybrid inference with GA-optimized parameters")

# Load optimized GA parameters (in case you want)
try:
    with open(os.path.join(RES_OUT, "optimized_ga_parameters.json"), "r") as f:
        opt = json.load(f)
    alpha_opt = opt["alpha"]
    beta_opt  = opt["beta"]
    gamma_opt = opt["gamma"]
    W_low_opt = opt["W_low"]
    W_med_opt = opt["W_med"]
    W_high_opt = opt["W_high"]
    print("Loaded optimized GA parameters.")
except:
    print("Using current variables (from Step 13).")

# --------------------------------------------
# Hybrid score function
# --------------------------------------------

def hybrid_score_vector(i, alpha, beta, gamma, Wl, Wm, Wh):
    """
    Computes hybrid ML + fuzzy + embedding score vector
    for rare disease index i against ALL common diseases.
    """

    sims = cos_rc[i]                   # cosine similarities
    fuzzy_r_numeric = rare_fuzzy_vec[i]
    emb_norm = np.linalg.norm(B2[i])   # embedding norm

    # recompute fuzzy using optimized weights
    mu_low, mu_med, mu_high = fuzzy_memberships(fuzzy_r_numeric)
    fuzzy_weighted = mu_low*Wl + mu_med*Wm + mu_high*Wh

    # final hybrid score
    hybrid = alpha * sims + beta * fuzzy_weighted + gamma * emb_norm
    return hybrid


# --------------------------------------------
# Compute hybrid matches for every rare disease
# --------------------------------------------

final_hybrid = {}

for i, rname in enumerate(rare_idx_names):

    hvec = hybrid_score_vector(
        i,
        alpha_opt, beta_opt, gamma_opt,
        W_low_opt, W_med_opt, W_high_opt
    )

    # choose top 5 closest common diseases
    top_idx = np.argsort(hvec)[::-1][:5]
    final_hybrid[rname] = [
        (common_idx_names[j], float(hvec[j])) for j in top_idx
    ]

print("Hybrid inference completed.")
print("Examples:", list(final_hybrid.items())[:2])

# Save to JSON
OUT_FILE = os.path.join(RES_OUT, "final_hybrid_zero_shot.json")
with open(OUT_FILE, "w") as f:
    json.dump(final_hybrid, f, indent=2)

print("Saved hybrid predictions to:", OUT_FILE)


In [ ]:
# ============================================
# AUTO-GENERATE SYSTEM LABELS (RARE + COMMON)
# ============================================

def infer_system(disease_name):
    """
    A simple rule-based mapping from disease name → high-level clinical system.
    This ensures every disease is assigned to a consistent system category.
    """
    name = disease_name.lower()

    if any(k in name for k in ["cardio", "heart", "vascular", "coronary"]):
        return "cardiovascular"
    if any(k in name for k in ["neuro", "brain", "nerv", "seizure", "cerebral"]):
        return "neurological"
    if any(k in name for k in ["hepato", "liver", "biliary"]):
        return "hepatic"
    if any(k in name for k in ["renal", "kidney", "nephro"]):
        return "renal"
    if any(k in name for k in ["gastro", "enter", "intestinal", "stomach", "colon", "gut"]):
        return "gastrointestinal"
    if any(k in name for k in ["immune", "immuno"]):
        return "immune"
    if any(k in name for k in ["respir", "lung", "pulmo", "asthma"]):
        return "respiratory"
    if any(k in name for k in ["muscle", "myo", "skeletal", "joint", "bone"]):
        return "musculoskeletal"
    if any(k in name for k in ["endocrine", "thyroid", "adrenal", "hormone"]):
        return "endocrine"
    if any(k in name for k in ["derma", "skin", "cutaneous"]):
        return "dermatological"
    if any(k in name for k in ["hemat", "blood", "erythro", "thrombo"]):
        return "hematological"
    if any(k in name for k in ["eye", "ocular", "vision", "retina"]):
        return "ocular"
    if any(k in name for k in ["ear", "hearing", "oto"]):
        return "auditory"

    # Default fallback for ambiguous names
    return "multisystem"


# These MUST match what your evaluation code expects
rare_system_label = {d: infer_system(d) for d in rare_idx_names}
common_system_label = {d: infer_system(d) for d in common_idx_names}

print("✔ System labels created successfully.")
print("\nSample rare disease system labels:")
print(list(rare_system_label.items())[:5])
print("\nSample common disease system labels:")
print(list(common_system_label.items())[:5])


In [ ]:
# ============================================
# STEP 15 — Evaluation & Visualisation
# ============================================

print("Running Step 15: Evaluation plots (t-SNE, Heatmap)")

# --------------------------------------------
# t-SNE on disease embeddings
# --------------------------------------------

all_vecs = np.vstack([A2, B2])
labels   = ["common"]*len(A2) + ["rare"]*len(B2)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
T = tsne.fit_transform(all_vecs)

plt.figure(figsize=(10,7))
plt.scatter(T[:len(A2),0], T[:len(A2),1], c='blue', label="Common Diseases", alpha=0.7)
plt.scatter(T[len(A2):,0], T[len(A2):,1], c='red',  label="Rare Diseases", alpha=0.7)
plt.legend()
plt.title("t-SNE of Common & Rare Disease Embeddings", fontsize=14)
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.grid(True)
plt.show()


# --------------------------------------------
# Heatmap of rare → common cosine similarity
# --------------------------------------------

sub_r = min(40, len(rare_idx_names))
sub_c = min(40, len(common_idx_names))

plt.figure(figsize=(12,8))
sns.heatmap(
    cos_rc[:sub_r, :sub_c],
    cmap='viridis',
    xticklabels=False,
    yticklabels=False
)
plt.title("Rare → Common Cosine Similarity Heatmap (subset)")
plt.xlabel("Common Diseases (subset)")
plt.ylabel("Rare Diseases (subset)")
plt.show()

print("Evaluation complete.")


In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

print("Running Step 16: Ranking evaluation metrics")

# --------------------------------------------------------
# Helper: compute Hit@K
# --------------------------------------------------------

def hit_k(true_idx, predicted_indices, k):
    return 1.0 if true_idx in predicted_indices[:k] else 0.0

# --------------------------------------------------------
# Helper: compute Reciprocal Rank (RR)
# --------------------------------------------------------

def reciprocal_rank(true_idx, predicted_indices):
    for rank, idx in enumerate(predicted_indices, start=1):
        if idx == true_idx:
            return 1.0 / rank
    return 0.0

# --------------------------------------------------------
# Build a synthetic "ground truth"
# --------------------------------------------------------
# For evaluation, we choose the MOST SIMILAR common disease
# (by cosine similarity) as a pseudo ground truth label.

pseudo_true_labels = {}

for i, rname in enumerate(rare_idx_names):
    sims = cos_rc[i]
    best_idx = int(np.argmax(sims))
    pseudo_true_labels[rname] = best_idx

print("Pseudo ground-truth mapping computed.")


# --------------------------------------------------------
# Evaluate hybrid predictions against pseudo-true labels
# --------------------------------------------------------

hit1 = 0
hit3 = 0
hit5 = 0
mrr  = 0
ndcg_list = []

for i, rname in enumerate(rare_idx_names):

    # Get true label
    true_idx = pseudo_true_labels[rname]

    # Get HYBRID predictions (indices)
    preds = [ common_idx_names.index(x[0]) for x in final_hybrid[rname] ]

    # compute metrics
    hit1 += hit_k(true_idx, preds, 1)
    hit3 += hit_k(true_idx, preds, 3)
    hit5 += hit_k(true_idx, preds, 5)
    mrr  += reciprocal_rank(true_idx, preds)

    # for NDCG
    relevance = np.zeros(len(common_idx_names))
    relevance[true_idx] = 1  # relevance 1 for true label
    scores = np.zeros(len(common_idx_names))
    scores[preds] = np.linspace(1, 0.1, len(preds))  # descending score
    ndcg_list.append(ndcg_score([relevance], [scores]))

# Normalize
N = len(rare_idx_names)
hit1 /= N
hit3 /= N
hit5 /= N
mrr  /= N
ndcg = np.mean(ndcg_list)

print("==========================================")
print("   RANKING METRICS (Hybrid Model)")
print("==========================================")
print(f"Hit@1 : {hit1:.4f}")
print(f"Hit@3 : {hit3:.4f}")
print(f"Hit@5 : {hit5:.4f}")
print(f"MRR   : {mrr:.4f}")
print(f"NDCG  : {ndcg:.4f}")
print("==========================================")

# Save to file
metrics = {
    "Hit@1": hit1,
    "Hit@3": hit3,
    "Hit@5": hit5,
    "MRR": mrr,
    "NDCG": ndcg
}

with open(os.path.join(RES_OUT, "hybrid_ranking_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved hybrid ranking metrics.")


In [ ]:
# ================================================
# STEP 17 — Automatic Clinical Body-System Mapping
# ================================================

print("Building clinical system classification...")

import re

# Define regex-based mapping rules
CLINICAL_SYSTEM_RULES = {
    "respiratory": [
        "asthma", "pneumonia", "bronch", "lung", "copd", "respirat"
    ],
    "gastrointestinal": [
        "gastr", "enter", "hepat", "colitis", "bowel", "abdominal"
    ],
    "infectious": [
        "infection", "viral", "bacterial", "parasit", "sepsis", "fever"
    ],
    "neurological": [
        "neuro", "brain", "seizure", "migraine", "spondylosis", "paral"
    ],
    "cardiovascular": [
        "cardio", "heart", "cardiac", "angina", "hypertension"
    ],
    "musculoskeletal": [
        "arthritis", "spond", "joint", "muscle", "bone", "fracture"
    ],
    "dermatologic": [
        "derma", "skin", "eczema", "rashes", "dermat"
    ],
    "genitourinary": [
        "cystitis", "renal", "kidney", "urinary", "neph"
    ],
    "endocrine_metabolic": [
        "diabetes", "thyroid", "adrenal", "metab"
    ],
    "immunological": [
        "immune", "autoimmune", "immun", "lymph"
    ],
    "hematologic": [
        "anemia", "hemoglobin", "hemat"
    ]
}

def map_to_system(disease_name):
    d = disease_name.lower()
    for system, keywords in CLINICAL_SYSTEM_RULES.items():
        for kw in keywords:
            if kw in d:
                return system
    return "multisystem"   # default fallback

# Map common diseases to systems
common_system_map = {d: map_to_system(d) for d in common_idx_names}

# Map rare diseases to systems
rare_system_map   = {d: map_to_system(d) for d in rare_idx_names}

print("Clinical system mapping complete.")


In [ ]:
# ====================================================
# STEP 18 — Evaluate Hybrid Model Using Body Systems
# ====================================================

print("Evaluating hybrid model using clinical systems...")

system_hit1 = 0
system_hit3 = 0
system_hit5 = 0
total = len(rare_idx_names)

detailed_results = {}

for r in rare_idx_names:
    rare_sys = rare_system_map[r]
    preds = final_hybrid[r]
    pred_systems = [ common_system_map[p[0]] for p in preds ]

    # Check matches
    sys1 = int(pred_systems[0] == rare_sys)
    sys3 = int(rare_sys in pred_systems[:3])
    sys5 = int(rare_sys in pred_systems[:5])

    system_hit1 += sys1
    system_hit3 += sys3
    system_hit5 += sys5

    detailed_results[r] = {
        "rare_system": rare_sys,
        "predicted_systems": pred_systems,
        "Hit@1": sys1,
        "Hit@3": sys3,
        "Hit@5": sys5
    }

# normalize
system_hit1 /= total
system_hit3 /= total
system_hit5 /= total

print("============================================")
print("     CLINICAL SYSTEM MATCHING RESULTS")
print("============================================")
print(f"System-Hit@1 : {system_hit1:.4f}")
print(f"System-Hit@3 : {system_hit3:.4f}")
print(f"System-Hit@5 : {system_hit5:.4f}")
print("============================================")

# save results
with open(os.path.join(RES_OUT, "clinical_system_evaluation.json"), "w") as f:
    json.dump(detailed_results, f, indent=2)

with open(os.path.join(RES_OUT, "clinical_system_metrics.json"), "w") as f:
    json.dump({
        "SystemHit@1": system_hit1,
        "SystemHit@3": system_hit3,
        "SystemHit@5": system_hit5
    }, f, indent=2)

print("Saved clinical system evaluation metrics.")


In [ ]:
# =============================================
# PLOT 1 — System Hit@K Bar Chart
# =============================================

import matplotlib.pyplot as plt

print("Plotting System Hit@K bar chart...")

hit_scores = [system_hit1, system_hit3, system_hit5]
labels = ["Hit@1", "Hit@3", "Hit@5"]

plt.figure(figsize=(7,5))
plt.bar(labels, hit_scores, color=["#4C72B0", "#55A868", "#C44E52"])
plt.ylim(0,1)
plt.ylabel("Score")
plt.title("Clinical System-Level Accuracy (Hybrid Model)")

for i,v in enumerate(hit_scores):
    plt.text(i, v+0.02, f"{v:.2f}", ha="center")

plt.show()


In [ ]:
# =================================================
# PLOT 2 — Clinical System Confusion Matrix
# =================================================

import seaborn as sns
import numpy as np
from collections import Counter

print("Building confusion matrix...")

systems = sorted(list(set(common_system_map.values()) | set(rare_system_map.values())))
sys_index = {s:i for i,s in enumerate(systems)}

# initialize confusion matrix
conf_mat = np.zeros((len(systems), len(systems)))

for r in rare_idx_names:
    r_sys = rare_system_map[r]
    preds = final_hybrid[r]
    pred_sys = common_system_map[preds[0][0]]
    conf_mat[sys_index[r_sys]][sys_index[pred_sys]] += 1

plt.figure(figsize=(12,10))
sns.heatmap(conf_mat, annot=True, fmt=".0f", xticklabels=systems, yticklabels=systems, cmap="Blues")
plt.xlabel("Predicted System")
plt.ylabel("Rare Disease System")
plt.title("Confusion Matrix — Clinical System Mapping")
plt.show()


In [ ]:
# =============================================
# PLOT 3 — t-SNE Visualization
# =============================================

from sklearn.manifold import TSNE

print("Running t-SNE...")

# build combined matrix
common_emb_list = np.vstack([common_embeddings[d] for d in common_idx_names])
rare_emb_list   = np.vstack([rare_embeddings[d] for d in rare_idx_names])

combined = np.vstack([common_emb_list, rare_emb_list])
labels = (["common"] * len(common_idx_names)) + (["rare"] * len(rare_idx_names))

tsne = TSNE(n_components=2, learning_rate="auto", init="random", perplexity=30)
tsne_coords = tsne.fit_transform(combined)

plt.figure(figsize=(8,8))
plt.scatter(tsne_coords[:len(common_idx_names),0], tsne_coords[:len(common_idx_names),1],
            label="Common Diseases", alpha=0.6, s=40)
plt.scatter(tsne_coords[len(common_idx_names):,0], tsne_coords[len(common_idx_names):,1],
            label="Rare Diseases", alpha=0.6, s=40)

plt.legend()
plt.title("t-SNE Visualization of Disease Embedding Space")
plt.show()


In [ ]:
# =========================================================
# PLOT 4 — Ablation Study (Cosine vs Hybrid vs Fuzzy)
# =========================================================

# Calculate fuzzy scores for common diseases using optimized weights for the ablation study
common_fuzzy_scores_optimized_list = []
for dname in common_idx_names:
    row_data = common_mat.loc[dname]
    common_fuzzy_scores_optimized_list.append(disease_fuzzy_score(row_data, W_low_opt, W_med_opt, W_high_opt))

# Create F_final matrix where each row is the vector of common disease fuzzy scores
# This means F_final[i,:] is identical for all i, representing the fuzzy score of common diseases.
F_final = np.tile(np.array(common_fuzzy_scores_optimized_list), (len(rare_idx_names), 1))

cos_hit1 = 0
fuzzy_hit1 = 0
hyb_hit1 = system_hit1

for i, r in enumerate(rare_idx_names):

    # true system
    true_sys = rare_system_map[r]

    # cosine top-1
    cos_best = common_idx_names[np.argmax(cos_rc[i])]
    if common_system_map[cos_best] == true_sys:
        cos_hit1 += 1

    # fuzzy top-1
    fuzzy_best = common_idx_names[np.argmax(F_final[i])]
    if common_system_map[fuzzy_best] == true_sys:
        fuzzy_hit1 += 1

# normalize
cos_hit1 /= len(rare_idx_names)
fuzzy_hit1 /= len(rare_idx_names)

plt.figure(figsize=(7,5))
plt.bar(["Cosine", "Fuzzy", "Hybrid"], [cos_hit1, fuzzy_hit1, hyb_hit1], color=["#4C72B0", "#C44E52", "#55A868"])
plt.ylim(0,1)
plt.ylabel("System-Hit@1")
plt.title("Ablation Study: Comparison of Scoring Methods")

for i,v in enumerate([cos_hit1, fuzzy_hit1, hyb_hit1]):
    plt.text(i, v+0.02, f"{v:.2f}", ha="center")

plt.show()

In [ ]:
# ==========================================
# USER INTERACTION: Query System
# ==========================================

import numpy as np

# --------------------------------------------------------
# Helper: calculate hybrid score for manual symptom input
# --------------------------------------------------------
def hybrid_score_manual(symptom_vector):
    """Compute hybrid similarity of a manual symptom input vs all common diseases."""

    # embed using MLP
    with torch.no_grad():
        x = torch.tensor(symptom_vector, dtype=torch.float32).unsqueeze(0)
        _, emb = net(x)
        emb = emb.squeeze(0).numpy()

    # Calculate fuzzy score for manual input
    # Convert symptom_vector (numpy array) to pandas Series for disease_fuzzy_score
    # Assuming 0/1 symptom vector means probability is 0/1 for each symptom
    manual_symptom_series = pd.Series(symptom_vector, index=common_symptoms_norm)
    # Use initial fuzzy weights to get the raw fuzzy score for the input
    fuzzy_r_manual = disease_fuzzy_score(manual_symptom_series, W_low_init, W_med_init, W_high_init)

    # Recompute fuzzy part with optimized weights for this manual input
    mu_low, mu_med, mu_high = fuzzy_memberships(fuzzy_r_manual)
    fuzzy_weighted_manual = mu_low*W_low_opt + mu_med*W_med_opt + mu_high*W_high_opt

    # Embedding norm for manual input
    emb_norm_manual = np.linalg.norm(emb)

    hybrid_scores = {}

    for idx, cname in enumerate(common_idx_names):

        # embedding similarity
        emb_common = common_embeddings[cname]
        sim_emb = cosine_similarity([emb], [emb_common])[0][0]

        # The hybrid score formula should match the one used in ga_fitness:
        # score = alpha_opt * sims + beta_opt * fuzzy_weighted + gamma_opt * emb_norm
        # Here, `sims` is `sim_emb` (pairwise embedding similarity),
        # `fuzzy_weighted` is `fuzzy_weighted_manual` (scalar for the input),
        # and `emb_norm` is `emb_norm_manual` (scalar for the input).
        score = (alpha_opt * sim_emb) + (beta_opt * fuzzy_weighted_manual) + (gamma_opt * emb_norm_manual)
        hybrid_scores[cname] = score

    # sort
    sorted_scores = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_scores[:5]


# --------------------------------------------------------
# MAIN INTERACTIVE FUNCTION
# --------------------------------------------------------

def predict():
    print("========================================")
    print(" AI RARE → COMMON DISEASE PREDICTOR")
    print("========================================")
    print("Choose mode:")
    print("1. Predict using a RARE DISEASE name")
    print("2. Predict using SYMPTOMS manually")

    mode = input("Enter 1 or 2: ")

    # ---------------------------------------------------
    # MODE 1: RARE DISEASE NAME
    # ---------------------------------------------------
    if mode == "1":
        print("\nAvailable rare diseases: (showing first 20)")
        print(rare_idx_names[:20])

        rname = input("\nEnter rare disease name exactly: ").strip()

        if rname not in rare_idx_names:
            print("❌ Rare disease not found.")
            return

        print("\n========================================")
        print(f" Results for Rare Disease: {rname}")
        print("========================================")

        preds = final_hybrid[rname]  # list of (disease, score)

        rare_sys = rare_system_map[rname]

        print(f"\nClinical System: {rare_sys}")
        print("\nTop-5 Most Similar Common Diseases:\n")

        for rank, (cname, s) in enumerate(preds[:5], start=1):
            print(f"{rank}. {cname}  (score={s:.4f}, system={common_system_map[cname]})łoś")

    # ---------------------------------------------------
    # MODE 2: MANUAL SYMPTOM ENTRY
    # ---------------------------------------------------
    elif mode == "2":

        print("\nEnter symptoms separated by commas.")
        print("Example: fever, abdominal pain, fatigue")

        user_symptoms = input("\nSymptoms: ").lower()

        symptom_list = user_symptoms.split(",")

        # create zero-vector
        symptom_vector = np.zeros(len(common_symptoms_norm))

        # fill vector
        for s in symptom_list:
            norm = normalize_symptom_name(s.strip())
            if norm in common_symptoms_norm: # Fixed: Used common_symptoms_norm directly
                idx = common_symptoms_norm.index(norm)
                symptom_vector[idx] = 1
            else:
                print(f"⚠️ Symptom '{s}' not recognized, skipping.")

        # compute predictions
        print("\nComputing predictions...\n")
        predictions = hybrid_score_manual(symptom_vector)

        print("========================================")
        print("TOP-5 COMMON DISEASE MATCHES")
        print("========================================")

        for rank, (cname, score) in enumerate(predictions, start=1):
            print(f"{rank}. {cname}  (score={score:.4f}, system={common_system_map[cname]})łoś")

    else:
        print("❌ Invalid option.")


# --------------------------------------------------------
# RUN INTERACTIVE MODE
# --------------------------------------------------------

predict()

In [ ]:
# ============================================
# HYBRID MODEL WITHOUT GA
# ============================================

print("Running Hybrid (No GA) Evaluation")

# Set fixed weights (you can tune manually)
alpha = 0.5    # cosine similarity weight
beta  = 0.5    # fuzzy score weight
gamma = 0.0    # embedding norm weight (optional)

# Use the default fuzzy membership weights
W_low  = 0.2
W_med  = 0.5
W_high = 0.8


In [ ]:
# Compute fuzzy score for each rare disease using default weights

def disease_fuzzy_score(row, Wl, Wm, Wh):
    total = 0.0
    for p in row.values:
        mu_low, mu_med, mu_high = fuzzy_memberships(p)
        total += mu_low*Wl + mu_med*Wm + mu_high*Wh
    return total

rare_fuzzy_vec = np.array([
    disease_fuzzy_score(rare_mat.loc[d], W_low, W_med, W_high)
    for d in rare_idx_names
])


In [ ]:
# Build hybrid predictions without GA

hybrid_no_ga = {}

for i, rname in enumerate(rare_idx_names):

    sims = cos_rc[i]                         # cosine B2[i] vs A2
    fuzzy_score = rare_fuzzy_vec[i]          # fuzzy score
    emb_norm = np.linalg.norm(B2[i])         # optionally used

    # Compute hybrid vector
    mu_low, mu_med, mu_high = fuzzy_memberships(fuzzy_score)
    fuzzy_weighted = mu_low*W_low + mu_med*W_med + mu_high*W_high

    hybrid_vec = alpha * sims + beta * fuzzy_weighted + gamma * emb_norm

    # Take top 5 common diseases
    top_idx = np.argsort(hybrid_vec)[::-1][:5]
    hybrid_no_ga[rname] = [
        (common_idx_names[j], float(hybrid_vec[j]))
        for j in top_idx
    ]

print("Hybrid (no GA) prediction dictionary created.")


In [ ]:
# Build hybrid predictions without GA

hybrid_no_ga = {}

for i, rname in enumerate(rare_idx_names):

    sims = cos_rc[i]                         # cosine B2[i] vs A2
    fuzzy_score = rare_fuzzy_vec[i]          # fuzzy score
    emb_norm = np.linalg.norm(B2[i])         # optionally used

    # Compute hybrid vector
    mu_low, mu_med, mu_high = fuzzy_memberships(fuzzy_score)
    fuzzy_weighted = mu_low*W_low + mu_med*W_med + mu_high*W_high

    hybrid_vec = alpha * sims + beta * fuzzy_weighted + gamma * emb_norm

    # Take top 5 common diseases
    top_idx = np.argsort(hybrid_vec)[::-1][:5]
    hybrid_no_ga[rname] = [
        (common_idx_names[j], float(hybrid_vec[j]))
        for j in top_idx
    ]

print("Hybrid (no GA) prediction dictionary created.")


In [ ]:
# ============================================
# PERFORMANCE EVALUATION (NO GA)
# ============================================

hit1 = 0
hit3 = 0
hit5 = 0

total = len(rare_idx_names)

for r in rare_idx_names:

    rare_sys = rare_system_label[r]
    preds = hybrid_no_ga[r]

    top5 = [p[0] for p in preds[:5]]
    top3 = top5[:3]
    top1 = top5[0]

    # Hit@1
    if common_system_label[top1] == rare_sys:
        hit1 += 1

    # Hit@3
    for p in top3:
        if common_system_label[p] == rare_sys:
            hit3 += 1
            break

    # Hit@5
    for p in top5:
        if common_system_label[p] == rare_sys:
            hit5 += 1
            break

# Print results ("tit values")
print("\n=========== HYBRID (NO GA) PERFORMANCE ===========")
print(f"Hit@1: {hit1}/{total} = {hit1/total:.4f}")
print(f"Hit@3: {hit3}/{total} = {hit3/total:.4f}")
print(f"Hit@5: {hit5}/{total} = {hit5/total:.4f}")
print("==================================================")
